<a href="https://colab.research.google.com/github/Aarchi01/Text-Summarization-Using-Bi-LSTM-and-BART/blob/main/AarchiGandhi_colabFile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text Summarization**

---


using Bi-Directional LSTM + Attention Mechanism and BART

**Dataset:**

This project uses ***CNN/DailyMail*** dataset ***(version 3.0.0)*** from *Hugging Face Datasets Library*.
This version of dataset is a fully-anonymized version which means it does NOT have the id column (retains original names and does not give entity) and only has Article and Highlights, making it suitable for real-world NLP applications.It comes with standardized train/test/validation splits.

Rationale behind using hugging face library:
* Manual loading of cnn_dailymail CSV dataset resulted in extreme data loss. only 4775 samples were loading with a split of (1414,1732,1629) for training, validation and testing. There were parsing issues which was leading to skipped rows.

* Using Hugging Face Datasets library: It becomes easy to access the dataset for others to reproduce. Also the library handles chaching and downloading automatically, making it efficient.

**Link**: https://huggingface.co/datasets/abisee/cnn_dailymail

The link above is for the general version of the cnn_dailymail dataset. Here, I am accessing the version 3.0.0 of the dataset.


**Dataset Structure:**
*   Articles - news articles from CNN and Daily Mail
*   Highlights - human written short summaries (abstractive)

**Dataset Statistics of version 3.0.0 are: **
*   Volume - around 300,000 unique article + summary pairs
*   Artice token count - 781 words
*   Summary token count - 56 words
*   Training samples - 287,113
*   Validation samples - 13,368
*   Test samples - 11,490

















**Approach for Text Summarization:**



*1. Data Preparation*
* Load the libraries and the dataset
* Pre-processing the text (cleaning, handling missing values, tokenising, padding)
* Creating decoder inputs with start tokens

*2. Defining the Model Architecture*
*  buliding seq2seq model with bi-directional LSTM and Attention mechanism
*  Dropout for regularization
*  Compiling model with Adam optimizer and sparse categorical crossentropy loss

*3. Training and Fine-tuning*
*   Setting early stopping & model checkpoint
*   Training the model
*   Evaluating and visualizing training metrics













# Data Loading / Acquisition

**The Libraries used along with their purpose:**

* pd - pandas for data manipulation
* np - numpy for mathematical operations
* re - regular expression (regex) for string manipulation
* nltk - natural language toolkit for analysing textual data
* stopwords - for removing the stopwords for pre-processing (the, and, is, etc)
* tokenize - importing word_tokenizer to break text into individual tokens
* WordNetLemmetizer - for bringing words to thier base form
* tf - tensorflow library
* Tokenizer - class of tf API, for converting text to sequences
* pad_sequences - for padding them 0 for same length
* plt - Matplotlib for visualization
* pickle - for saving the models
* os - for handling file paths and directories



In [ ]:
#importing libraries
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize as tokenize
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import matplotlib.pyplot as plt
import pickle
import os

#download nltk resources
nltk.download('punkt') # for tokenizing text into words/sentences
nltk.download('stopwords') #list of stopwords to remove
nltk.download('wordnet') #base form of words

np.random.seed(42) #for reproducibility

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
#installing the latest version of the datasets library
!pip install --upgrade datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 18.7 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system 

In [ ]:
#importing and loading the dataset of version 3.0.0 from Hugging Face library
from datasets import load_dataset
dataset = load_dataset("cnn_dailymail", "3.0.0")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

In [ ]:
# Convert to pandas DataFrame
train_df = pd.DataFrame({
    'article': dataset['train']['article'],
    'highlights': dataset['train']['highlights']
})

val_df = pd.DataFrame({
    'article': dataset['validation']['article'],
    'highlights': dataset['validation']['highlights']
})

test_df = pd.DataFrame({
    'article': dataset['test']['article'],
    'highlights': dataset['test']['highlights']
})

# Check dataset info
print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")

Training set shape: (287113, 2)
Validation set shape: (13368, 2)
Test set shape: (11490, 2)


In [ ]:
#checking missing values
print('Missing values in training dataset:\n')
print(train_df.isnull().sum())

Missing values in training dataset:

article       0
highlights    0
dtype: int64


# **Data Cleaning & Pre-Processing**

The below code performs text cleaning and pre-processing tasks.

The cleaning function involves removing the HTML tags, any special characters and stopwords, converting to lower case, tokenizing and lemmetizing.

Due to computational constraint and time-sensitive nature of the project, I have used sub-sample of the dataset for faster training for all the 3 sets by using sample() method. The samples are set to 1000 for training and 500 for validation and 200 for test sets.

Now, just to be sure, I check for any missing values in the cleaned datasets and drop if any.

In [ ]:
#Text cleaning
#reference for writing: from lab sheet (lab NLP)
import nltk
nltk.download('punkt_tab')
stop_words = set(stopwords.words('english'))
#lemmatizer = WordNetLemmatizer()

def text_cleaned(text):
  """
  cleaning and pre-processing text by removing/handling html tags, special characters,
  lower case and lemmetization to bring all the words to it's root word.
  """
  lemmatizer = WordNetLemmatizer()

  if isinstance(text, str):
    text = re.sub(r'<.*?>', '', text)  #removing any HTML tags <>
    text = re.sub(r'[^A-Za-z0-9\s]', ' ', text) #removing the special characters
    text = text.lower() #lower-casing the text
    words = tokenize(text) #breaking the text into individual tokens

    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)
  else:
    return ''

#As the dataset is too large, I am using a subset / small part of dataset for training.
train_sample = 1000
val_sample = 500
test_sample = 200

#sample() here randomly selects a subset of the dataset, but not crossing the length of entire dataset.
#setting random_state ensures that, the code chooses the same subset everytime.
train_df = train_df.sample(min(train_sample, len(train_df)), random_state=42)
val_df = val_df.sample(min(val_sample, len(val_df)), random_state=42)
test_df = test_df.sample(min(test_sample, len(test_df)), random_state=42)

print(f"Training set shape after sampling: {train_df.shape}")
print(f"Validation set shape after sampling: {val_df.shape}")
print(f"Test set shape after sampling: {test_df.shape}")

#applying the text_cleaned function on the datasets.
train_df['clean_article'] = train_df['article'].apply(text_cleaned)
train_df['clean_highlights'] = train_df['highlights'].apply(text_cleaned)
val_df['clean_article'] = val_df['article'].apply(text_cleaned)
val_df['clean_highlights'] = val_df['highlights'].apply(text_cleaned)
test_df['clean_article'] = test_df['article'].apply(text_cleaned)
test_df['clean_highlights'] = test_df['highlights'].apply(text_cleaned)

#if there are any missing values in any dataset, I want to remove them before further implementing solution
train_df.dropna(subset=['clean_article', 'clean_highlights'], inplace=True)
val_df.dropna(subset=['clean_article', 'clean_highlights'], inplace=True)
test_df.dropna(subset=['clean_article', 'clean_highlights'], inplace=True)

print(f"Training set shape after cleaning: {train_df.shape}")
print(f"Validation set shape after cleaning: {val_df.shape}")
print(f"Test set shape after cleaning: {test_df.shape}")




[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Training set shape after sampling: (1000, 2)
Validation set shape after sampling: (500, 2)
Test set shape after sampling: (200, 2)
Training set shape after cleaning: (1000, 4)
Validation set shape after cleaning: (500, 4)
Test set shape after cleaning: (200, 4)


In [ ]:
print(train_df.columns)

Index(['article', 'highlights', 'clean_article', 'clean_highlights'], dtype='object')


In [ ]:
#assigning the parameters before training
max_article = 200  #max words allowed in article input
max_highlight = 40 #max words allowed in highlights/summary
vocab_size = 5000 #size of the vocabulary for tokenizer
embedding_dim = 32 #dimension of the word embedding for vector representation
LSTM_units = 32 #number of neurons in the layer
batch_size = 16  #number of samples trained before weight changes
epochs = 10 #number of iterations

The next step will prepare the data for the model by tokenizing, converting text to numerical sequences and padding the sequences.

1. Two different tokenizers are created for two features and are given the limit of the vocab_size. The words that lie outside of this size are replaced with Out-of-vocabulary tokens (OOV).

2. Each text is converted into sequence of numbers, because padding can only be applied on numerical data and not on words or text.

3. Sequences are standardized to fixed lengths, specified in the code performed above. Shorter sequences will be padded with 0 in the end whereas the longer ones will be cut at the end.


In [ ]:
#tokenizing and padding articles and summaries
tokenizer_article = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer_article.fit_on_texts(list(train_df['clean_article']) + list(val_df['clean_article']))    #fit_on_texts() -> builds vocabulary and assigns a unique integer value to frequently occurring words.
tokenizer_highlight = Tokenizer(num_words=vocab_size, oov_token='<OOV>')
tokenizer_highlight.fit_on_texts(list(train_df['clean_highlights']) + list(val_df['clean_highlights']))

#converting articles to sequences before padding
train_article_to_seq = tokenizer_article.texts_to_sequences(train_df['clean_article'])          #texts_to_sequences() -> takes those values into consideration and converts them into numerical form for training.
val_article_to_seq = tokenizer_article.texts_to_sequences(val_df['clean_article'])
test_article_to_seq = tokenizer_article.texts_to_sequences(test_df['clean_article'])

#converting highlights/summaries to sequences
#where each token will be given a unique numerical id
train_highlight_to_seq = tokenizer_highlight.texts_to_sequences(train_df['clean_highlights'])
val_highlight_to_seq = tokenizer_highlight.texts_to_sequences(val_df['clean_highlights'])
test_highlight_to_seq = tokenizer_highlight.texts_to_sequences(test_df['clean_highlights'])

#padding the article sequences
train_article_pad = pad_sequences(train_article_to_seq, maxlen=max_article, padding='post', truncating='post')    #pad_sequences() -> ensures that all sequences are of same length.
val_article_pad = pad_sequences(val_article_to_seq, maxlen=max_article, padding='post', truncating='post')
test_article_pad = pad_sequences(test_article_to_seq, maxlen=max_article, padding='post', truncating='post')

#padding the highlights sequences
train_highlight_pad = pad_sequences(train_highlight_to_seq, maxlen=max_highlight, padding='post', truncating='post')
val_highlight_pad = pad_sequences(val_highlight_to_seq, maxlen=max_highlight, padding='post', truncating='post')
test_highlight_pad = pad_sequences(test_highlight_to_seq, maxlen=max_highlight, padding='post', truncating='post')

print(f"Training set shape after tokenization and padding: {train_article_pad.shape}")
print(f"Validation set shape after tokenization and padding: {val_article_pad.shape}")
print(f"Test set shape after tokenization and padding: {test_article_pad.shape}")



Training set shape after tokenization and padding: (1000, 200)
Validation set shape after tokenization and padding: (500, 200)
Test set shape after tokenization and padding: (200, 200)


In [ ]:
print(train_highlight_pad.shape)
print(val_highlight_pad.shape)
print(test_highlight_pad.shape)

(1000, 40)
(500, 40)
(200, 40)


# **Defining Model Architecture**

This is a ***Bi-LSTM based seq2seq, encoder + decoder model architecture with attention***.

Bi-directional LSTMs is used as they are proficient in handling long-text well and understand the context of the text more accurately by captueing most of the semantics from both the sides.

The encoder Bi-LSTM takes the article as input and produces hidden states for each and every single input token. The decoder LSTM will generate summary one token at a time. I am applying a shifting operation which creates a '*teacher-forcing*' approach. In this approach, the decoder's first input is a *start* token. It ensures that, during training, each predicted token becomes the input for the next steps.

The ***Attention*** will put more weight on more relevant/important words of the encoder output.

Here, the decoder input will be shifted right by 1 position with a start token for both training and validation sets.
Decoder output will have the original summary sequence.


In [ ]:
#creating decoder inputs for training
train_decoder_input = np.zeros_like(train_highlight_pad) #creating the array of same shape
train_decoder_input[:, 1:] = train_highlight_pad[:, :-1] #shifting one position right
train_decoder_input[:, 0] = 1  #start token

val_decoder_input = np.zeros_like(val_highlight_pad)
val_decoder_input[:, 1:] = val_highlight_pad[:, :-1]
val_decoder_input[:, 0] = 1  #start token

test_decoder_input = np.zeros_like(test_highlight_pad)
test_decoder_input[:, 1:] = test_highlight_pad[:, :-1]
test_decoder_input[:, 0] = 1  #start token

print('Train Decoder Input Shape: ',train_decoder_input.shape)
print('Val Decoder Input Shape: ',val_decoder_input.shape)
print('Test Decoder Input Shape: ',test_decoder_input.shape)

#assigning vocabulary sizes(counts the number of unique tokens and adds 1 for padding token(index 0))
'''
tokenizer_article.word_index will map words to their index values which starts from 1,
thus we add 1 to get the correct number of unique tokens.
'''
article_vocab_size = min(len(tokenizer_article.word_index) + 1, vocab_size)
highlight_vocab_size = min(len(tokenizer_highlight.word_index) + 1, vocab_size)

print(f"Article Vocabulary Size: {article_vocab_size}")
print(f"Highlight Vocabulary Size: {highlight_vocab_size}")

Train Decoder Input Shape:  (1000, 40)
Val Decoder Input Shape:  (500, 40)
Test Decoder Input Shape:  (200, 40)
Article Vocabulary Size: 5000
Highlight Vocabulary Size: 5000


**Model Definition**

* importing necessary tf libraries
* Defining the **seq2seq model function**
* Encoder - consisting of input layer, embedding layer, bidirectional LSTM, combines both the hidden states
* Decoder - consisting of decoder inputs(shifted), embedding layer, LSTM layer
* Attention Mechanism - consisting of attention layer, context vector (weighted sum) and combines decoder output with context vector for comprehensive information.
* Output layer - Dense layer with **softmax** activation and **dropout** to prevent overfitting.
* Compiling model - consisting of **Adam Optimizer**, **Sparse categorical crossentropy as loss function** and **Accuracy for metrics**.
* Building model - creating the model instance and printing the model summary.

In [ ]:
#defining the DL Model
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout, Concatenate, Attention
from tensorflow.keras.optimizers import Adam

def seq2seq_model(article_vocab_size, highlight_vocab_size, embedding_dim, LSTM_units, max_article, max_highlight, dropout_rate=0.2):
  #Encoder
  inputs = Input(shape=(max_article,), name='inputs')

  #encoder embedding
  #converts words to dense vector representations before passing to lstm
  embedding = Embedding(input_dim = article_vocab_size, output_dim = embedding_dim,
                        input_length = max_article, name='embedding')(inputs)

  #Bi-LSTM for Encoder
  bi_lstm = Bidirectional(LSTM(LSTM_units,
                               return_sequences=True, #outputs entire sequence
                               return_state=True, # returns the hidden states
                               dropout=dropout_rate, #reduces overfitting
                               name='bi_lstm_encoder'))
  encoder_outputs, forward_h, forward_c, backward_h, backward_c = bi_lstm(embedding)
  #because Bi-LSTM captures data from both forward and backward direction for better understanding.

  #join / concatenate both states (forward + backword) for decoder
  states = [Concatenate()([forward_h, backward_h]),
            Concatenate()([forward_c, backward_c])]


  #Decoder
  inputs_decoder = Input(shape=(max_highlight,), name='inputs_decoder')

  #decoder embedding
  #converts words to dense vector representations
  embedding_decoder = Embedding(input_dim = highlight_vocab_size, output_dim = embedding_dim,
                                input_length = max_highlight, name='embedding_decoder')(inputs_decoder)

  #Decoder LSTM (keeping states as initial states)
  lstm = LSTM(LSTM_units*2,  #double hidden units for better learning capacity
              return_sequences=True, #outputs entire sequence
              return_state=True, #return hidden and all other states
              dropout=dropout_rate, #reduces overfitting
              name='lstm')
  decoder_outputs, _, _ = lstm(embedding_decoder, initial_state=states)

  #Attention Mechanism, puts attention weights to most important words
  attention = Attention(name='attention')
  context_vector = attention([decoder_outputs, encoder_outputs])

  #join the context vector with the decoder output
  decoder_context = Concatenate(axis=-1, name='concat')([decoder_outputs, context_vector])

  #Dropout for Regularisation
  decoder_context = Dropout(dropout_rate)(decoder_context)

  #Output Layer
  #Softmax activation function here, ensures that the model outputs the word based on its probaility. this way the model will learn better gradually.
  decoder_outputs = Dense(highlight_vocab_size, activation='softmax', name='outputs')(decoder_context)

  #Model
  model = Model(inputs=[inputs, inputs_decoder], outputs=decoder_outputs)

  model.compile(optimizer=Adam(learning_rate=0.0005),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
  return model

#build Model
model = seq2seq_model(
    article_vocab_size=article_vocab_size,
    highlight_vocab_size=highlight_vocab_size,
    embedding_dim=embedding_dim,
    LSTM_units=LSTM_units,
    max_article=max_article,
    max_highlight=max_highlight
)

model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, 200)       │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 200, 32)   │    160,000 │ inputs[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ inputs_decoder      │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_7     │ [(None, 200, 64), │     16,640 │ embedding[0][0]   │
│ (Bidirectional)     │ (None, 32),       │            │                   │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32),       │            │                   │
│                     │ (None, 32)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_decoder   │ (None, 40, 32)    │    160,000 │ inputs_decoder[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_14      │ (None, 64)        │          0 │ bidirectional_7[… │
│ (Concatenate)       │                   │            │ bidirectional_7[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_15      │ (None, 64)        │          0 │ bidirectional_7[… │
│ (Concatenate)       │                   │            │ bidirectional_7[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 40, 64),  │     24,832 │ embedding_decode… │
│                     │ (None, 64),       │            │ concatenate_14[0… │
│                     │ (None, 64)]       │            │ concatenate_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 40, 64)    │          0 │ lstm[0][0],       │
│ (Attention)         │                   │            │ bidirectional_7[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concat              │ (None, 40, 128)   │          0 │ lstm[0][0],       │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 40, 128)   │          0 │ concat[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ outputs (Dense)     │ (None, 40, 5000)  │    645,000 │ dropout_7[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,006,472 (3.84 MB)

 Trainable params: 1,006,472 (3.84 MB)

 Non-trainable params: 0 (0.00 B)

# **Training and Fine-Tuning**

In [ ]:
#Define callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('seq2seq_model.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

In [ ]:
#Reshape the target sequences to have dimensions for sparse categorical crossentropy.
# Without reshaping, the loss function will try to compare (batch_size, length, vocab_Size)
#with my targets (batch_size, length),which would give dimension mismatch error.
train_highlight_target = train_highlight_pad.reshape(train_highlight_pad.shape[0], train_highlight_pad.shape[1], 1)
val_highlight_target = val_highlight_pad.reshape(val_highlight_pad.shape[0], val_highlight_pad.shape[1], 1)



**Implementing training with a few different experiments for better performance and finding the best model**

In [ ]:
!pip install rouge_score

In [ ]:
from tensorflow.keras.optimizers import Adam, RMSprop, SGD, Adagrad
from rouge_score import rouge_scorer
import numpy as np
import datetime
import time

# Define a function to evaluate ROUGE scores
def compute_rouge_scores(predictions, references):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    # Convert predictions and references to strings before scoring
    scores = []
    for ref, pred in zip(references, predictions):
        # Flatten and convert to strings
        ref_str = " ".join(map(str, ref.flatten()))
        pred_str = " ".join(map(str, pred.flatten()))

        scores.append(scorer.score(ref_str, pred_str))

    avg_scores = {
        'rouge1': np.mean([score['rouge1'].fmeasure for score in scores]),
        'rouge2': np.mean([score['rouge2'].fmeasure for score in scores]),
        'rougeL': np.mean([score['rougeL'].fmeasure for score in scores])
    }

    return avg_scores

from tensorflow.keras.callbacks import ModelCheckpoint

# run_experiment function to include optimizers
def run_experiment(params, train_data, val_data, test_data, experiment_name):
    print(f"\n\n{'='*50}")
    print(f"Starting experiment: {experiment_name}")
    print(f"Parameters: {params}")
    print(f"{'='*50}\n")

    # Extracting parameters from previous code
    article_vocab_size = params.pop('article_vocab_size')
    highlight_vocab_size = params.pop('highlight_vocab_size')
    max_article = params.pop('max_article')
    max_highlight = params.pop('max_highlight')
    epochs = params.pop('epochs')
    batch_size = params.pop('batch_size')
    learning_rate = params.pop('learning_rate')
    optimizer_name = params.pop('optimizer')

    # Selecting optimizer based on the experiment configuration
    if optimizer_name == 'adam':
        optimizer = Adam(learning_rate=learning_rate)
    elif optimizer_name == 'sgd':
        optimizer = SGD(learning_rate=learning_rate)
    elif optimizer_name == 'adagrad':
        optimizer = Adagrad(learning_rate=learning_rate)

    # Extracting the data
    train_article_pad, train_decoder_input, train_highlight_target = train_data
    val_article_pad, val_decoder_input, val_highlight_target = val_data
    test_article_pad, test_decoder_input, test_highlight_target = test_data

    # Recording start time
    start_time = time.time()

    # Model creation
    def create_model(params, optimizer):
    # Encoder
      inputs = Input(shape=(params['max_article'],), name='inputs')

      # encoder embedding
      embedding = Embedding(input_dim=params['article_vocab_size'], output_dim=params['embedding_dim'],
                              input_length=params['max_article'], name='embedding')(inputs)

      # Bi-LSTM for Encoder
      dropout_rate = params.get('dropout', 0.2)
      bi_lstm = Bidirectional(LSTM(params['LSTM_units'],
                                      return_sequences=True,
                                      return_state=True,
                                      dropout=dropout_rate,
                                      name='bi_lstm_encoder'))
      encoder_outputs, forward_h, forward_c, backward_h, backward_c = bi_lstm(embedding)

      # join / concatenate both states (forward + backward)
      states = [Concatenate()([forward_h, backward_h]),
                  Concatenate()([forward_c, backward_c])]

      # Decoder
      inputs_decoder = Input(shape=(params['max_highlight'],), name='inputs_decoder')

      # decoder embedding
      embedding_decoder = Embedding(input_dim=params['highlight_vocab_size'], output_dim=params['embedding_dim'],
                                      input_length=params['max_highlight'], name='embedding_decoder')(inputs_decoder)

      # Decoder LSTM
      lstm = LSTM(params['LSTM_units']*2,
                    return_sequences=True,
                    return_state=True,
                    dropout=dropout_rate,
                    name='lstm')
      decoder_outputs, _, _ = lstm(embedding_decoder, initial_state=states)

      # Attention Mechanism
      attention = Attention(name='attention')
      context_vector = attention([decoder_outputs, encoder_outputs])

      # join the context vector with the decoder output
      decoder_context = Concatenate(axis=-1, name='concat')([decoder_outputs, context_vector])

      # Dropout for Regularization
      decoder_context = Dropout(dropout_rate)(decoder_context)

      # Output Layer
      decoder_outputs = Dense(params['highlight_vocab_size'], activation='softmax', name='outputs')(decoder_context)

      # Model
      model = Model(inputs=[inputs, inputs_decoder], outputs=decoder_outputs)

      model.compile(optimizer=optimizer,
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])
      return model

      # Defining the ModelCheckpoint callback to save the best model in SavedModel format
    checkpoint_path = f"best_model_{experiment_name}.keras"
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True),
        ModelCheckpoint(checkpoint_path, monitor='val_loss', save_best_only=True, mode='min', verbose=1)
    ]

    # Training model
    history = model.fit(
        [train_article_pad, train_decoder_input], train_highlight_target,
        validation_data=([val_article_pad, val_decoder_input], val_highlight_target),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks
    )

    # Calculate training time
    training_time = time.time() - start_time

    # Evaluating model on the test data
    test_predictions = model.predict([test_article_pad, test_decoder_input])
    test_predictions = np.argmax(test_predictions, axis=-1)

    # Computing ROUGE scores for the predictions
    references = test_highlight_target  # Actual highlights from the test set
    rouge_scores = compute_rouge_scores(test_predictions, references)

    # Extracting results
    results = {
        'experiment_name': experiment_name,
        'optimizer': optimizer_name,
        'embedding_dim': params['embedding_dim'],
        'LSTM_units': params['LSTM_units'],
        'dropout': params.get('dropout', 0.2),
        'batch_size': batch_size,
        'learning_rate': learning_rate,
        'epochs_completed': len(history.history['loss']),
        'final_train_loss': history.history['loss'][-1],
        'final_train_acc': history.history['accuracy'][-1],
        'final_val_loss': history.history['val_loss'][-1],
        'final_val_acc': history.history['val_accuracy'][-1],
        'rouge1': rouge_scores['rouge1'],
        'rouge2': rouge_scores['rouge2'],
        'rougeL': rouge_scores['rougeL'],
        'training_time_seconds': training_time
    }

    print(f"\nExperiment completed in {training_time:.2f} seconds")
    print(f"Final results: Train Loss: {results['final_train_loss']:.4f}, Train Acc: {results['final_train_acc']:.4f}")
    print(f"             Val Loss: {results['final_val_loss']:.4f}, Val Acc: {results['final_val_acc']:.4f}")
    print(f"             ROUGE-1: {results['rouge1']:.4f}, ROUGE-2: {results['rouge2']:.4f}, ROUGE-L: {results['rougeL']:.4f}")

    return results, model, history

# the run_experiments function
def run_experiments():
    # Preparing data
    train_highlight_target = train_highlight_pad.reshape(train_highlight_pad.shape[0], train_highlight_pad.shape[1], 1)
    val_highlight_target = val_highlight_pad.reshape(val_highlight_pad.shape[0], val_highlight_pad.shape[1], 1)
    test_highlight_target = test_highlight_pad.reshape(test_highlight_pad.shape[0], test_highlight_pad.shape[1], 1)

    train_data = (train_article_pad, train_decoder_input, train_highlight_target)
    val_data = (val_article_pad, val_decoder_input, val_highlight_target)
    test_data = (test_article_pad, test_decoder_input, test_highlight_target)

    # Configurations for experiments with different optimizers
    experiment_configs = [
        {
            'experiment_name': 'adam_optimizer',
            'article_vocab_size': article_vocab_size,
            'highlight_vocab_size': highlight_vocab_size,
            'embedding_dim': 64,
            'LSTM_units': 64,
            'dropout': 0.3,
            'batch_size': 32,
            'max_article': max_article,
            'max_highlight': max_highlight,
            'epochs': 10,
            'learning_rate': 0.001,
            'optimizer': 'adam'
        },

        {
            'experiment_name': 'sgd_optimizer',
            'article_vocab_size': article_vocab_size,
            'highlight_vocab_size': highlight_vocab_size,
            'embedding_dim': 64,
            'LSTM_units': 64,
            'dropout': 0.3,
            'batch_size': 32,
            'max_article': max_article,
            'max_highlight': max_highlight,
            'epochs': 10,
            'learning_rate': 0.001,
            'optimizer': 'sgd'
        },
        {
            'experiment_name': 'adagrad_optimizer',
            'article_vocab_size': article_vocab_size,
            'highlight_vocab_size': highlight_vocab_size,
            'embedding_dim': 64,
            'LSTM_units': 64,
            'dropout': 0.3,
            'batch_size': 32,
            'max_article': max_article,
            'max_highlight': max_highlight,
            'epochs': 10,
            'learning_rate': 0.001,
            'optimizer': 'adagrad'
        }
    ]

    # Storing results
    all_results = []

    # Creating timestamp for saving results
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    # Running experiments
    for config in experiment_configs:
        experiment_name = config['experiment_name']
        results, model, history = run_experiment(config.copy(), train_data, val_data, test_data, experiment_name)
        all_results.append(results)

        # Save the model and other outputs as before

    # Final results
    final_df = pd.DataFrame(all_results)
    print("\n\nFinal Results:")
    print(final_df[['experiment_name', 'final_train_loss', 'final_val_loss', 'final_train_acc', 'final_val_acc', 'rouge1', 'rouge2', 'rougeL', 'training_time_seconds']])

    return final_df

# Running all experiments
results_df = run_experiments()




Starting experiment: adam_optimizer
Parameters: {'experiment_name': 'adam_optimizer', 'article_vocab_size': 5000, 'highlight_vocab_size': 5000, 'embedding_dim': 64, 'LSTM_units': 64, 'dropout': 0.3, 'batch_size': 32, 'max_article': 200, 'max_highlight': 40, 'epochs': 10, 'learning_rate': 0.001, 'optimizer': 'adam'}

Epoch 1/10
31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1873 - loss: 8.4883
Epoch 1: val_loss improved from inf to 8.03362, saving model to best_model_adam_optimizer.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 5s 61ms/step - accuracy: 0.1905 - loss: 8.4838 - val_accuracy: 0.2082 - val_loss: 8.0336
Epoch 2/10
30/32 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2713 - loss: 7.4676
Epoch 2: val_loss improved from 8.03362 to 6.32271, saving model to best_model_adam_optimizer.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.2709 - loss: 7.4245 - val_accuracy: 0.2105 - val_loss: 6.3227
Epoch 3/10
31/32 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2642 - loss: 5.93

In conclusion,

due to time and resources constraint, I could only perform these many combinations of fine tuning, as it was taking a long time to learn.
According to a research made in 2021 by a team of people on the same process (Bi-LSTM + Attention), it took a minimum of 40 epochs to train and achieve the accuracy of 93.6%.

Here, In my approach, after experimenting with a few hyperparameters (Optimizers), and training for 10 epochs, I got comparatively very less accuracy and rougue scores, specifically, the best one was with Adam Optimizer with rouge scores of **40.58**, **26.52** and **40.58** respectively and accuracy of **34.46%**, which with more samples, epochs and training would result in much higher scores and accuracy.
Another approach for the Summarization task for a better Text Summarizer, I decided to fine-tune a pre-trained model called **BART** (Bidirectional AutoRegressive Transformer) by *facebook* to train the **CNN/DailyMail Dataset from the hugging face library**.

For Real-world use of the model, I have then **deployed the app on Hugging Face Spaces using Gradio**.



> *The steps to access the link to my web app are described at the end of the colab file.*





---



# **Using Bidirectional AutoRegressive Transformer (BART)**

Here, I am building a Text Summarizer using BART (Bidirectional Autoregressive Transformer) which will help to generate a short summary of the long time consuming documents.

The whole process is divided into **2 sections**:

*data loading, pre-processing and training*
*loading model and launching web app via Gradio*.
let us first install the transformer and datasets because I am using the version 3.0.0 of the cnn/dailymails dataset from the hugging face library.

# Data Loading, Pre-processing and Training

In [ ]:
!pip install transformers datasets

The next step, I have imported the necessary libraries as follows:

* BartTokenizer - it will tokenize the text and feed into the model

* BartForConditionalGeneration - this is a BART model class for doing tasks like summarization

* TrainingArguments, Trainer - these are used for fine-tuning and training the model.

* DataCollatorForSeq2Seq - this will help prepare correct batch sizes for the task for faster training.

I have then, mounted my drive and saved the fine-tuned model and tokenizer there for future use.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from datasets import load_dataset
import os
from google.colab import drive

# Mount Google Drive to save your model persistently
drive.mount('/content/drive')

# Create a directory in your Google Drive to save the model
save_directory = '/content/drive/MyDrive/summarizer_model'
os.makedirs(save_directory, exist_ok=True)

The below code is loading the pre-trained BART model.

In [ ]:
# Load dataset
dataset = load_dataset('cnn_dailymail', '3.0.0')
model_name = 'facebook/bart-large-cnn'
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

In [ ]:
# Define preprocessing function
'''
this code will take article and highlight text as input and target and set a max length of tokens.
truncation = true refers to cutting the longer sentences and padding means adding tokens
to shorter sentences for equal length.
'''
def preprocess(example):
  inputs = tokenizer(example['article'], max_length=512, truncation=True, padding='max_length')
  targets = tokenizer(example['highlights'], max_length=128, truncation=True, padding='max_length')
  inputs['labels'] = targets['input_ids'] # (assigns token ids of highlights to input), what should the prediction be based on the article
  return inputs

# Preprocess and prepare data
tokenized_dataset = dataset.map(preprocess, batched=True, remove_columns=dataset['train'].column_names)
train_dataset = tokenized_dataset['train'].select(range(1000))
val_dataset = tokenized_dataset['validation'].select(range(200))

In [ ]:
# Set up training arguments
training_args = TrainingArguments(
    output_dir=save_directory + '/checkpoints',
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8, #accumulates gradients over 8 steps before updating for simulating larger batches
    evaluation_strategy='epoch',
    save_strategy='epoch',
    num_train_epochs=1,
    learning_rate=3e-5,
    fp16=True, #with this the training is faster and less memory-intensive on GPU
    report_to='none', #does not report to external platforms like tensorboard
    logging_dir=save_directory + '/logs',

In [ ]:
# Set up trainer and train
'''
Collator automatically pads sequences, assigns weight (attention) and assembles them into a batch.
'''
from transformers import DataCollatorForSeq2Seq
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

trainer.train()

# Save the fine-tuned model to Google Drive
model_save_path = save_directory + '/bart-summarizer-cnn'
model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"Model saved to {model_save_path}")

# Test saving/loading worked by loading and testing
test_model = BartForConditionalGeneration.from_pretrained(model_save_path)
test_tokenizer = BartTokenizer.from_pretrained(model_save_path)
print("Successfully loaded saved model and tokenizer!")

# Loading model and Launching Gradio

When you want access the public URL for the deployed model, you only have to run the codes in this section which loads the fine-tuned model saved on my drive and launches the app.

In [ ]:
!pip install gradio

In the code below, I first load my fine-tuned model from my drive and for security concerns, check if it available in the drive. If under any scenario, the model does not exist, it will continue using the pre-trained 'facebook/bart-large-cnn' model instead.

In [ ]:
from transformers import BartTokenizer, BartForConditionalGeneration
import gradio as gr
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Path to your saved model
model_path = '/content/drive/MyDrive/summarizer_model/bart-summarizer-cnn'

# Check if model exists
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model = BartForConditionalGeneration.from_pretrained(model_path)
    tokenizer = BartTokenizer.from_pretrained(model_path)
    print("Model loaded successfully!")
else:
    print(f"Model not found at {model_path}")
    print("Loading pre-trained model instead...")
    model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn')
    tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')

Mounted at /content/drive
Loading model from /content/drive/MyDrive/summarizer_model/bart-summarizer-cnn


/usr/local/lib/python3.12/dist-packages/transformers/models/bart/configuration_bart.py:177: UserWarning: Please make sure the config includes `forced_bos_token_id=0` in future versions. The config can simply be saved and uploaded again to be fixed.
  warnings.warn(


OSError: Error no file named pytorch_model.bin, model.safetensors, tf_model.h5, model.ckpt.index or flax_model.msgpack found in directory /content/drive/MyDrive/summarizer_model/bart-summarizer-cnn.

In [ ]:
# Define the summarization function to use the fine-tuned BART model to generate summaries
'''
return_tensors=pt means that the output will be a PyTorch tensor which is the expected format for BART.
num_beams = 4, means the model will look for top 4 most likely next words
in the sequence based on overall probability.
length penalty will tell the model to not generate very large summary and improve the quality.
'''
def summarize_article(article, creativity):
    inputs = tokenizer(article, max_length=512, return_tensors='pt', truncation=True)
    summary_ids = model.generate(
        inputs['input_ids'],
        max_length=160,
        min_length=60,
        do_sample=True,
        temperature=creativity,
        top_k=50,
        top_p=0.95,
        num_beams=4,
        repetition_penalty=1.2,
        early_stopping=True
    )
    '''
    ids[0] will take the first generated summary out of 4 beam results
    skip_special_tokens will skip the padding tokens from the final summary.
    '''
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

In [ ]:
# Create and launch Gradio interface
'''
gr.interface() will create a gradio interface object, which will have my model and text (textbox) for
input and output, and title and description for the interface.
when you run interface.launch, colab will generate a public URL to access the interface.
'''
interface = gr.Interface(
    fn=summarize_article,
    inputs=[gr.Textbox(label="Enter Article"),
            gr.Slider(0.5,1.2, value=0.9, label="Creativity Level")],
    outputs='text',
    title='Abstractive Article Summarizer',
    description='Enter an article to get a summary and adjust creativity to control paraphrasing strength'
)

interface.launch()

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b760e23a6e82668a59.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Conclusion:

This model effectively generates a concise summary based on the input text.

However, te gradio link has its own limitation. The link will only work when the file is running.

To overcome this limitation,
I published my app on **Hugging Face Spaces**.
It is a platform which provides a space to publish or deploy our models with ease so that every person can access it without running the code again and again.

To do so, I first created an account on Hugging Face and then created a new space. I then chose 'Gradio' and named my application as '**Text Summarizer**'.

The next step was connecting the space with my model. For that, I created 2 files namely, **app.py** (my model code) and **requirements.txt**. After this the space will be generated in a few minutes and start running and get public.

The Link:

https://aarchigandhi-text-summarizer.hf.space/?__theme=system



> Note: if the space is inactive for some days, it will go in sleeping mode, all one has to do to get it running again is click of '***restart space***' on the screen and it will start running in around 2 minutes.

